# exp7_reference_policies — politiques de référence

Trois politiques sans apprentissage bornent les résultats : aléatoire, toujours exploiter,
toujours s'abstenir. Comme elles n'apprennent pas, leurs valeurs découlent de la seule
dynamique de la ressource (méthode journal, *Reference policies*). Elles sont donc calculées
directement par Monte Carlo sur 5 000 épisodes, sans le pipeline d'entraînement, et sans
aucun fichier de résultats.

Tentative réussie avec une probabilité $R_t / R_0$, $R_{t+1} = \max(0, R_t - C_t)$, fin de
l'épisode à l'épuisement ou après $T_{\max}$ pas.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO = Path("..").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from main_gt_ais import INITIAL_RESOURCES, MAX_STEPS, NB_AGENTS, gini

EPISODES = 5000
SEED = 0

In [ ]:
def simulate(attempt_probability, episodes=EPISODES, seed=SEED):
    """all episodes in parallel: each agent attempts with this probability at each step."""
    rng = np.random.default_rng(seed)
    resource = np.full(episodes, float(INITIAL_RESOURCES))
    steps = np.zeros(episodes, dtype=int)
    meals = np.zeros((episodes, NB_AGENTS))
    active = np.ones(episodes, dtype=bool)
    for _ in range(MAX_STEPS):
        idx = np.flatnonzero(active)
        if idx.size == 0:
            break
        attempts = rng.random((idx.size, NB_AGENTS)) < attempt_probability
        success = attempts & (rng.random((idx.size, NB_AGENTS)) < resource[idx, None] / INITIAL_RESOURCES)
        meals[idx] += success
        resource[idx] = np.maximum(0.0, resource[idx] - success.sum(axis=1))
        steps[idx] += 1
        active[idx] = resource[idx] > 0
    return {"steps_mean": steps.mean(), "steps_sd": steps.std(ddof=1),
            "censored": (active & (steps >= MAX_STEPS)).mean(),
            "meals_per_episode": meals.sum(axis=1).mean(),
            "gini": np.mean([gini(m) for m in meals])}


table = pd.DataFrame({"random": simulate(0.5), "always_exploit": simulate(1.0),
                      "always_abstain": simulate(0.0)}).T
display(table.round(3))

## Contrôle : valeurs annoncées dans la méthode

Si la dynamique change, cette cellule échoue et le paragraphe de la méthode est à mettre à
jour.

In [ ]:
stated = {"always_exploit": {"steps_mean": 563, "steps_sd": 105, "gini": 0.036},
          "random": {"steps_mean": 975, "steps_sd": 51, "censored": 0.71, "gini": 0.045}}
tolerance = {"steps_mean": 5, "steps_sd": 5, "censored": 0.03, "gini": 0.003}
for policy, values in stated.items():
    for name, value in values.items():
        assert abs(table.loc[policy, name] - value) <= tolerance[name], (policy, name, table.loc[policy, name])
assert table.loc["always_abstain", "steps_mean"] == MAX_STEPS and table.loc["always_abstain", "meals_per_episode"] == 0
print("les valeurs de la méthode sont retrouvées")